# 9d · Judge benchmark — how good is the grounding judge?

nb9b runs a judge over a whole split and prunes what it rejects. That number is only worth what the
judge is worth, and a judge cannot be validated on its own output: it grounds ~95% of what it sees,
so a sample of its verdicts contains almost no true negatives.

This notebook scores the judge against **500 gold-labelled assertions** drawn from the
gemini-2.5-flash validation graphs:

| half | how it was made | gold |
|---|---|---|
| 250 **graph** rows | drawn as-is, proportional per article | hand-labelled |
| 250 **corrupted** rows | one deterministic meaning-changing edit each, verified absent from the article | not supported, by construction |

**The judge under test is `grounding_core`'s** — the same prompt, verbalisers and quote check nb9b
runs. Edit the prompt there and re-run this notebook; that loop is the point of splitting the two.

### What the last run found (2026-08-19, qwen3-32b-think, 3 repeats)

- recall on unsupported **75–77%**, false-positive rate **1.3–3.1%**
- but recall on the **22 real extractor errors** was only **23–41%** — the synthetic half is easier
  than reality, because real errors here are wrong *properties*, not wrong entities
- judged grounding rate on real rows **95.2%** vs human **91.2%** → the run's rate is a **ceiling**
- pairwise agreement between identical runs **94.4–96.8%** → **~5% of verdicts flip between runs**,
  so no judge-vs-judge difference below ~4 points means anything

The prompt has since been given an explicit "the property and types are part of the claim" clause,
aimed squarely at that miss population. Re-running this notebook is how you find out whether it worked.

In [1]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [2]:
# ---- WHOSE GRAPHS THE BENCHMARK IS BUILT FROM -----------------------------------------------
# The 500 items were drawn from this tree, so the benchmark measures judges on ITS assertions.
# Judging another model's graphs is an extrapolation until a second benchmark is built from those.
RESULTS_MODEL = "gemini-2.5-flash"
TRACK_DIR     = "e2e"
GRAPH_NAME    = "nary_graph_dedup_main_val_r1_extended_consolidated.json"

# ---- WHICH JUDGES TO TEST -------------------------------------------------------------------
# One entry per judge. `repeats` > 1 runs the SAME seed several times: that measures re-runnability
# (vLLM batches continuously, so a seed makes a run repeatable, not determined) and gives the noise
# floor every comparison below has to clear.
#
# Each vLLM judge needs its own local port. With both models already served on the cluster, open a
# tunnel per node and give each judge its `url` — then this cell runs straight through:
#     ./cluster/labia-tunnel.sh <qwen-node>                  -> localhost:8000
#     PORT=8001 ./cluster/labia-tunnel.sh <llama-node>       -> localhost:8001
# Name the node (or job id) explicitly for BOTH: with two servers up, the no-arg form is ambiguous.
# Section 2 then runs the judges one after another; section 2b runs them concurrently over one
# thread pool, each worker bound to its own client. With both servers up, 2b is the one you want.
#
# Repeat BOTH judges, not just one — their noise floors need not be alike. qwen3-32b-think is forced
# to temperature 0.6, so its repeats vary by sampling AND by vLLM's batching; llama33-70b runs greedy
# with no seed, so its repeats vary by batching alone. The gap between the two spreads tells you how
# much of qwen's instability is temperature and how much is the server.
DEFAULT_URL = "http://localhost:8000/v1"
JUDGES = [
    {"backend": "qwen3-32b-think", "repeats": 3, "url": "http://localhost:8000/v1"},
    {"backend": "llama33-70b",     "repeats": 3, "url": "http://localhost:8001/v1"},
]

# $BENCH_JUDGES replaces the list wholesale, as a JSON array, so a headless run can add or swap
# judges without editing this cell. Unset leaves the two above untouched.
#   BENCH_JUDGES='[{"backend":"deepseek-v4-flash","repeats":3}]'
# HOSTED judges (Fireworks, DeepSeek) must NOT carry a "url": make_llm resolves their endpoint from
# BACKENDS, and a localhost url would send them down the vLLM tunnel instead. Section 2b already
# skips any judge whose verdict file exists, so re-running never overwrites verdicts already scored.
import os as _os, json as _json
_bj = _os.environ.get("BENCH_JUDGES")
if _bj:
    JUDGES = _json.loads(_bj)
    print(f"JUDGES overridden from $BENCH_JUDGES: "
          + ", ".join(f"{j['backend']}x{j.get('repeats', 1)}" for j in JUDGES))

BENCH_SEED = 1234
BENCH_BATCH = 20        # same batch for every judge, so batch length is not a confound
WORKERS    = 4          # articles in flight; keep IDENTICAL across repeats or you change the batching

from pathlib import Path
RESULTS_DIR = ROOT / "results" / RESULTS_MODEL
ARTICLE_DIR = ROOT / "dataset" / "articles"
GLOBAL_DIR  = RESULTS_DIR / "_global"          # the benchmark lives here

import re as _re
_natkey = lambda s: [int(t) if t.isdigit() else t for t in _re.split(r"(\d+)", str(s))]
ARTICLE_IDS = sorted((p.name for p in RESULTS_DIR.iterdir()
                      if p.is_dir() and (p / TRACK_DIR / GRAPH_NAME).exists()), key=_natkey)
print(f"{len(ARTICLE_IDS)} article(s) available | benchmark dir {GLOBAL_DIR.relative_to(ROOT)}")
print("judges:", ", ".join(f"{j['backend']}x{j.get('repeats', 1)}" for j in JUDGES))

JUDGES overridden from $BENCH_JUDGES: deepseek-v4-flashx3
20 article(s) available | benchmark dir results/gemini-2.5-flash/_global
judges: deepseek-v4-flashx3


In [3]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None
LAST_USAGE = None

_RETRY_STATUS = (429, 500, 502, 503, 504)
_RETRY_WORDS = ("UNAVAILABLE", "RESOURCE_EXHAUSTED", "INTERNAL", "high demand", "overloaded")


# A 429 is retryable when it means "too fast" and pointless when it means "out of money": backing
# off five times against a depleted balance just burns three minutes before failing anyway.
_FATAL_WORDS = ("credits are depleted", "billing", "quota exceeded for quota metric",
                "exceeded your current quota", "insufficient")


def _gemini_retryable(exc) -> bool:
    """A transient server-side failure, as opposed to one that repeating cannot fix."""
    msg = str(exc).lower()
    if any(w in msg for w in _FATAL_WORDS):
        return False
    code = getattr(exc, "code", None) or getattr(exc, "status_code", None)
    if code in _RETRY_STATUS:
        return True
    return any(w.lower() in msg for w in _RETRY_WORDS)


def _gemini_call_with_retry(fn, attempts=5, base=4):
    """Call `fn()`, retrying TRANSIENT API errors with backoff (4s, 8s, 16s, 32s).

    Without this one `503 UNAVAILABLE - This model is currently experiencing high demand` ends the
    whole stage: it killed nb4 two articles into twenty and took nb6b and nb7b down with it. Google's
    own message says such spikes are temporary, which is what a backoff is for. A 400 or 403 still
    fails at once — repeating a malformed or unauthorised request achieves nothing.
    """
    import time
    for attempt in range(attempts):
        try:
            return fn()
        except Exception as exc:                       # noqa: BLE001 - re-raised below
            if attempt == attempts - 1 or not _gemini_retryable(exc):
                raise
            wait = min(60, base * 2 ** attempt)
            code = getattr(exc, "code", None) or getattr(exc, "status_code", "?")
            print(f"    [retry {attempt + 1}/{attempts - 1}] {type(exc).__name__} {code} — "
                  f"waiting {wait}s", flush=True)
            time.sleep(wait)


def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object, recording tokens + latency."""
    global last_latency_s, LAST_USAGE
    t0 = time.perf_counter()
    resp = _gemini_call_with_retry(lambda: client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    ))
    last_latency_s = time.perf_counter() - t0
    LAST_USAGE = nb_parallel.record_gemini_usage(resp, last_latency_s)   # prompt/output/THINKING tokens + latency
    return json.loads(resp.text)

# --- imports only; the backend is selected per judge in section 2 ----------------------------
# This notebook switches judges inside a loop, so it must NOT bind one here. The Gemini client
# above stays defined because `activate("gemini-2.5-flash", ...)` keeps whatever `gemini_json`
# the notebook already has — that is how the control arm works.
import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend, grounding_core as gc, judge_benchmark as jb
for _m in (nb_parallel, llm_backend, gc, jb):
    importlib.reload(_m)
nb_parallel.install()

import json, hashlib
PROMPT_SHA = hashlib.sha1(gc.GROUNDING_SYSTEM_PROMPT.encode()).hexdigest()[:12]

def load_graph(aid):   return json.loads((RESULTS_DIR / aid / TRACK_DIR / GRAPH_NAME).read_text())
def load_article(aid): return (ARTICLE_DIR / f"{aid}.txt").read_text()

print(f"judge under test: grounding_core prompt {PROMPT_SHA} ({len(gc.GROUNDING_SYSTEM_PROMPT)} chars)")

Note: you may need to restart the kernel to use updated packages.
judge under test: grounding_core prompt 67305d45abb6 (3112 chars)


## 1. The benchmark

Already built and hand-reviewed — this section is here so it can be rebuilt from scratch, and so you
can read a few corruptions before trusting them.

`build()` **refuses to overwrite** an existing sheet: rebuilding renumbers every item, which orphans
the labels and every verdict file scored against them. Pass `overwrite=True` only to start over.

In [4]:
# Rebuild only if you mean to. Normally you just look at what is there.
REBUILD = False

if REBUILD:
    graphs   = {a: load_graph(a) for a in ARTICLE_IDS}
    articles = {a: load_article(a) for a in ARTICLE_IDS}
    jb.build(graphs, articles, GLOBAL_DIR, gc.verbalise_node, gc.make_relation_verbaliser,
             gc.norm_text, n_clean=250, n_corrupt=250, source=GRAPH_NAME)

_items = jb.load_items(GLOBAL_DIR)
_meta  = jb.load_json(GLOBAL_DIR / jb.META_FILE)
_gold, _unlab, _failed, _key, _ = jb.gold_labels(GLOBAL_DIR)
print(f"{len(_items)} items | {sum(1 for x in _items if x['origin']=='graph')} graph / "
      f"{sum(1 for x in _items if x['origin']=='corrupted')} corrupted")
print(f"gold: {len(_gold)} — {sum(_gold.values())} supported / {len(_gold)-sum(_gold.values())} unsupported")
print(f"unlabelled {len(_unlab)} | failed corruptions excluded {len(_failed)}")
print(f"awaiting human review: {[x['index'] for x in _items if (x.get('label_source') or '').strip()] or 'none'}")
print("\ncorruption mix:", _meta["realised_mix"])

500 items | 250 graph / 250 corrupted
gold: 475 — 228 supported / 247 unsupported
unlabelled 0 | failed corruptions excluded 0
awaiting human review: [42, 47, 111, 124, 137, 170, 204, 235, 248, 283, 295, 312, 332, 348, 366, 386, 387, 393, 419]

corruption mix: {'object_swap': 55, 'property_swap': 45, 'number_edit': 40, 'direction_flip': 30, 'subject_swap': 30, 'hedge_flip': 25, 'type_change': 25}


In [5]:
# One example of each corruption operator — an edit that produced word salad would be a negative
# any judge catches for the wrong reason, and would inflate recall.
_by_i, _seen = {x["index"]: x for x in _items}, set()
for i, k in _key.items():
    if k["origin"] != "corrupt" or k["op"] in _seen:
        continue
    _seen.add(k["op"])
    print(f'\n[{i}] {k["article_id"]}  {k["op"]}   ({k["edit"]})')
    print("   before:", k["original_assertion"].splitlines()[0][:110])
    print("   after :", _by_i[int(i)]["assertion"].splitlines()[0][:110])


[0] art73  hedge_flip   (object_text: increase -> decrease)
   before: "Previous cyberattacks on Taiwan from China" (PreviousCyberattackStatistics) --[indicatedValue]--> "6% increas
   after : "Previous cyberattacks on Taiwan from China" --[indicatedValue]--> "6% decrease".

[1] art15  type_change   (type MuseumOpeningEvent -> Opinion)
   before: A event of type "MuseumOpeningEvent" exists, described as "Lalique Museum opened".
   after : An event exists, described as "Lalique Museum opened".

[2] art12  direction_flip   (reversed: Patriot systems <-> loitering munitions)
   before: "Patriot systems" (MilitarySystem) --[forPurposeOf]--> "loitering munitions" (WeaponType).
   after : "loitering munitions" --[forPurposeOf]--> "Patriot systems".

[3] art2  object_swap   (object "Music taste" -> "Historical Narratives")
   before: "Carbaug grew in music taste with Swift" (Opinion) --[about]--> "Music taste" (Concept).
   after : "Carbaug grew in music taste with Swift" --[about]--> "Histo

## 2. Run the judges

Each judge is activated in turn, then run over all 500 items grouped by article. One verdict file per
`<backend>_seed<seed>_run<n>` tag, so nothing overwrites anything and every run stays comparable.

The cell skips any tag that already has a verdict file, so it is safe to re-run after a crash — or
after serving a different model on the same port.

In [6]:
def run_bench(backend, tag, url=None, seed=BENCH_SEED, workers=WORKERS, batch=None):
    """Activate `backend`, judge all 500 items, write the verdict file. -> tag."""
    spec = dict(llm_backend.BACKENDS[backend])
    over = {}
    if spec.get("kind") != "gemini":
        # ALWAYS pin the URL. llm_backend keeps its base_url until something changes it, so a judge
        # that passes none inherits the previous judge's server — which, with two models tunnelled,
        # means silently benchmarking one model against the other's endpoint and getting a plausible
        # number back. Never let it default here.
        over["base_url"] = url or spec.get("url") or DEFAULT_URL
    if spec.get("kind") != "gemini" and seed is not None:
        over["seed"] = seed
    llm_backend.activate(backend, ROOT, globals(), results=RESULTS_MODEL, **over)
    if spec.get("kind") != "gemini":
        print(f"   -> {llm_backend.BASE_URL}  model={llm_backend.MODEL}")

    gc.set_llm(gemini_json)                       # whatever activate() just bound
    # Give every judge the SAME batch size when comparing them: batch length changes how much a
    # model has to hold in one answer, so a thinking judge at 20 versus a structured one at 60 is a
    # confound sitting inside the comparison. 20 suits both; it only costs more calls.
    gc.configure(judge_batch=batch or (20 if spec.get("thinking") else 60))

    items  = jb.load_items(GLOBAL_DIR)            # fresh dicts: no verdicts carried between runs
    groups = dict(jb.by_article(items))
    arts   = {a: load_article(a) for a in groups}

    def _one(aid, _g=groups, _a=arts):
        gc.set_article(_a[aid])                   # thread-local: the prompt is built from it
        nb_parallel.quiet(gc.run_judge, _g[aid], jb.assertion_of, aid)
        return sum(x.get("grounding", {}).get("supported") is not None for x in _g[aid])

    nb_parallel.reset_usage()
    nb_parallel.pmap(_one, list(groups), workers=workers)
    jb.save_verdicts(items, GLOBAL_DIR, tag, config={
        "backend": backend, "model": llm_backend.MODEL if spec.get("kind") != "gemini" else GEMINI_MODEL,
        **{k: v for k, v in spec.items() if k not in ("kind", "model", "url")},
        "temperature": llm_backend.TEMPERATURE, "top_p": llm_backend.TOP_P,
        "seed": llm_backend.SEED, "judge_batch": gc.JUDGE_BATCH, "workers": workers,
        "graph": GRAPH_NAME, "results_model": RESULTS_MODEL, "prompt_sha1": PROMPT_SHA})
    nb_parallel.usage_report(f"nb9d · {tag}", wall_seconds=nb_parallel.LAST_WALL_SECONDS,
                             write=GLOBAL_DIR / f"token_usage_judge_benchmark_{tag}.json")
    return tag

# The tag carries the PROMPT hash. Without it, re-running after a prompt edit would find the old
# verdict file, skip the run, and quietly compare a new-prompt judge against an old-prompt one —
# the exact silent error this benchmark exists to prevent.
for _j in JUDGES:
    for _r in range(1, _j.get("repeats", 1) + 1):
        _stem = f"{_j['backend']}_p{PROMPT_SHA}_seed{BENCH_SEED}"
        _tag = f"{_stem}_run{_r}" if _j.get("repeats", 1) > 1 else _stem
        if (GLOBAL_DIR / jb.VERDICTS_FILE.format(tag=_tag)).exists():
            print(f"{_tag}: already done, skipping"); continue
        print(f"\n=== {_tag} ===")
        run_bench(_j["backend"], _tag, url=_j.get("url"), batch=_j.get("batch", BENCH_BATCH))


=== deepseek-v4-flash_p67305d45abb6_seed1234_run1 ===
backend: HOSTED https://api.fireworks.ai/inference/v1 model=accounts/fireworks/models/deepseek-v4-flash-0731 thinking=False structured=True max_tokens=None temperature=caller top_p=None seed=1234 timeout=3600s params={'reasoning_effort': 'none'}  ->  results/gemini-2.5-flash/   (reading/writing another model's tree)
   -> https://api.fireworks.ai/inference/v1  model=accounts/fireworks/models/deepseek-v4-flash-0731
20 item(s), running on 4 threads

[1/20] art4  (12.0s)
[2/20] art2  (20.7s)
[3/20] art3  (24.5s)
[4/20] art1  (32.9s)
[5/20] art5  (26.1s)
[6/20] art12  (13.9s)
[7/20] art8  (18.4s)
[8/20] art15  (10.8s)
[9/20] art17  (5.8s)
[10/20] art44  (6.0s)
[11/20] art23  (10.1s)
[12/20] art46  (5.6s)
[13/20] art57  (5.5s)
[14/20] art55  (11.5s)
[15/20] art58  (8.5s)
[16/20] art52  (15.6s)
[17/20] art59  (13.9s)
[18/20] art72  (11.6s)
[19/20] art74  (8.9s)
[20/20] art73  (71.2s)

done — 20/20 ok in 129.0s
wrote judge_benchmark_verdi

### 2b. All judges at once (optional)

Section 2 activates one backend at a time, which is right when the models take turns on one port.
With a server per model already up and a tunnel to each, this runs them **concurrently** instead:
one thread pool over every (judge, run, article) task, each worker bound to its own client.

Two things make that safe, and neither was true before:
`llm_backend.make_llm` returns a client bound to one server rather than mutating module globals, and
`grounding_core.set_thread_llm` binds it to the calling thread. Without both, the second judge
activated would silently answer the first judge's calls and the numbers would still look plausible.

`WORKERS` here is the total in flight across **all** judges, so each server sees roughly
`WORKERS / len(JUDGES)` concurrent requests. Keep it the same as the sequential run if you want the
two comparable: concurrency changes vLLM's batching, which is one of the things being measured.

In [7]:
# --- prerequisites, so this cell can be run on its own -------------------------------------
# Everything below needs the config cell (JUDGES, GLOBAL_DIR, BENCH_*) to have run. The imports and
# PROMPT_SHA normally come from the client cell; they are repeated here because this is the cell you
# re-run after a tunnel drops, and jumping straight to it should not fail on a missing import.
import sys, pathlib, importlib, hashlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend, grounding_core as gc, judge_benchmark as jb
PROMPT_SHA = hashlib.sha1(gc.GROUNDING_SYSTEM_PROMPT.encode()).hexdigest()[:12]
if "load_article" not in globals():
    def load_article(aid): return (ARTICLE_DIR / f"{aid}.txt").read_text()

PARALLEL_WORKERS = 8            # total across all judges (~4 per server with two judges)

_pending = {}                    # tag -> (items, groups, client)
for _j in JUDGES:
    _client = llm_backend.make_llm(_j["backend"], base_url=_j.get("url"), seed=BENCH_SEED,
                                   gemini_fn=globals().get("gemini_json"))
    for _r in range(1, _j.get("repeats", 1) + 1):
        _stem = f"{_j['backend']}_p{PROMPT_SHA}_seed{BENCH_SEED}"
        _tag = f"{_stem}_run{_r}" if _j.get("repeats", 1) > 1 else _stem
        if (GLOBAL_DIR / jb.VERDICTS_FILE.format(tag=_tag)).exists():
            print(f"{_tag}: already done, skipping"); continue
        _items = jb.load_items(GLOBAL_DIR)          # a fresh copy per run: no verdicts shared
        _pending[_tag] = (_items, dict(jb.by_article(_items)), _client)
        print(f"queued {_tag}  ->  {_client.info.get('base_url', 'gemini')}")

_arts = {a: load_article(a) for a in ARTICLE_IDS}

# Interleave the judges. pmap dispatches in input order, so listing every task of one judge before
# the next means the pool works through the first judge entirely before touching the second — the
# run stays parallel across articles but goes sequential across judges, and one server sits idle.
# Round-robin by backend so the first N tasks spread across all of them.
from itertools import zip_longest
_by_backend = {}
for _tag, (_, _groups, _) in _pending.items():
    _by_backend.setdefault(_tag.split("_p")[0], []).extend((_tag, a) for a in _groups)
_tasks = [t for _row in zip_longest(*_by_backend.values()) for t in _row if t is not None]

def _judge_task(task):
    tag, aid = task
    items, groups, client = _pending[tag]
    gc.set_thread_llm(client)                       # THIS thread talks to THIS judge only
    gc.configure(judge_batch=BENCH_BATCH)
    gc.set_article(_arts[aid])
    nb_parallel.quiet(gc.run_judge, groups[aid], jb.assertion_of, f"{tag}/{aid}")
    return sum(x.get("grounding", {}).get("supported") is not None for x in groups[aid])

if _tasks:
    print(f"\n{len(_tasks)} tasks ({len(_pending)} runs x {len(_arts)} articles), "
          f"{PARALLEL_WORKERS} workers\n")
    nb_parallel.reset_usage()
    _res = nb_parallel.pmap(_judge_task, _tasks, workers=PARALLEL_WORKERS,
                            label=lambda t: f"{t[0].split('_p')[0]}/{t[1]}")
    _failed = [r.item for r in _res if not r.ok]
    if _failed:
        print(f"\n!! {len(_failed)} task(s) failed: {_failed[:6]} — re-run this cell to retry them")

    for _tag, (_items, _groups, _client) in _pending.items():
        _n = sum(1 for x in _items if x.get("grounding", {}).get("supported") is not None)
        if _n == 0:
            print(f"!! {_tag}: 0/{len(_items)} judged — NOT written. The backend was unreachable "
                  f"for the whole run; fix it and re-run this cell.")
            continue
        if _n < len(_items):
            print(f"!! {_tag}: only {_n}/{len(_items)} judged — written, but treat it as partial "
                  f"and re-run once the backend is healthy (delete the file first).")
        jb.save_verdicts(_items, GLOBAL_DIR, _tag, config={
            **_client.info, "judge_batch": BENCH_BATCH, "workers": PARALLEL_WORKERS,
            "graph": GRAPH_NAME, "results_model": RESULTS_MODEL, "prompt_sha1": PROMPT_SHA,
            "mode": "parallel"})
    nb_parallel.usage_report("nb9d · parallel benchmark", wall_seconds=nb_parallel.LAST_WALL_SECONDS,
                             write=GLOBAL_DIR / f"token_usage_judge_benchmark_parallel.json")
else:
    print("nothing to run — every tag already has a verdict file")

deepseek-v4-flash_p67305d45abb6_seed1234_run1: already done, skipping
deepseek-v4-flash_p67305d45abb6_seed1234_run2: already done, skipping
deepseek-v4-flash_p67305d45abb6_seed1234_run3: already done, skipping
nothing to run — every tag already has a verdict file


## 3. Score

`score()` reports, per run: recall on unsupported assertions (the headline), the false-positive rate
on real graph rows, precision, and **recall per corruption type** — which is where judges actually
differ. A judge can look strong overall and be blind to one class.

`compare()` puts every run side by side and prints **pairwise verdict agreement**. Two runs of one
model bound its reproducibility; two models are the real comparison. Read no difference smaller than
the same-model spread.

In [8]:
for _t in jb.verdict_tags(GLOBAL_DIR):
    jb.score(GLOBAL_DIR, _t, examples=0)
    print()

=== deepseek-v4-flash_p67305d45abb6_seed1234_run1 ===
scored 475 items  (247 gold-unsupported, 228 gold-supported)
  25 item(s) excluded as out of scope for grounding (type_change) — type quality is evaluated separately

  RECALL on unsupported   207/247 = 83.8% [79%, 88%]
  false-positive rate     8/228 = 3.5% [2%, 7%]   (good assertions the judge rejected)
  precision               207/215 = 96.3% [93%, 98%]
  F1                      0.896
  accuracy                427/475 = 89.9%

  recall by corruption type
    direction_flip   16/30  =  53.3% [36%, 70%]
    property_swap    33/45  =  73.3% [59%, 84%]
    hedge_flip       23/25  =  92.0% [75%, 98%]
    object_swap      55/55  = 100.0% [93%, 100%]
    number_edit      40/40  = 100.0% [91%, 100%]
    subject_swap     30/30  = 100.0% [89%, 100%]

  failure_type matches the edit's design: 132/197 = 67.0%   {'extrinsic->extrinsic': 79, 'intrinsic->intrinsic': 53, 'extrinsic->intrinsic': 39, 'intrinsic->extrinsic': 26}
  of the 40 missed

In [9]:
jb.compare(GLOBAL_DIR)


judge                                     n   recall     FPR    prec     acc
deepseek-v4-flash_p67305d45abb6_seed1234_run1  475    83.8%    3.5%   96.3%   89.9%
deepseek-v4-flash_p67305d45abb6_seed1234_run2  475    83.8%    5.7%   94.1%   88.8%
deepseek-v4-flash_p67305d45abb6_seed1234_run3  475    83.0%    3.1%   96.7%   89.7%
llama33-70b_p67305d45abb6_seed1234_run1  475    81.4%    3.5%   96.2%   88.6%
llama33-70b_p67305d45abb6_seed1234_run2  475    81.4%    3.5%   96.2%   88.6%
llama33-70b_p67305d45abb6_seed1234_run3  475    81.4%    3.5%   96.2%   88.6%
qwen3-32b-think_p67305d45abb6_seed1234_run1  475    84.6%    2.6%   97.2%   90.7%
qwen3-32b-think_p67305d45abb6_seed1234_run2  475    86.6%    2.2%   97.7%   92.0%
qwen3-32b-think_p67305d45abb6_seed1234_run3  475    85.8%    3.5%   96.4%   90.9%
qwen3-32b-think_seed1234_run1           475    76.5%    3.1%   96.4%   86.3%
qwen3-32b-think_seed1234_run2           475    79.4%    1.8%   98.0%   88.4%
qwen3-32b-think_seed1234_run3       

[{'tag': 'deepseek-v4-flash_p67305d45abb6_seed1234_run1',
  'n': 475,
  'tp': 207,
  'fn': 40,
  'fp': 8,
  'tn': 220,
  'recall': 0.8380566801619433,
  'fpr': 0.03508771929824561,
  'precision': 0.9627906976744186,
  'accuracy': 0.8989473684210526,
  'by_op': {'hedge_flip': {'caught': 23, 'n': 25},
   'direction_flip': {'caught': 16, 'n': 30},
   'object_swap': {'caught': 55, 'n': 55},
   'property_swap': {'caught': 33, 'n': 45},
   'number_edit': {'caught': 40, 'n': 40},
   'subject_swap': {'caught': 30, 'n': 30}},
  'unlabelled': 0,
  'failed_corruptions': 0,
  'unjudged': 0,
  'model_drafted_labels': 0},
 {'tag': 'deepseek-v4-flash_p67305d45abb6_seed1234_run2',
  'n': 475,
  'tp': 207,
  'fn': 40,
  'fp': 13,
  'tn': 215,
  'recall': 0.8380566801619433,
  'fpr': 0.05701754385964912,
  'precision': 0.9409090909090909,
  'accuracy': 0.888421052631579,
  'by_op': {'hedge_flip': {'caught': 23, 'n': 25},
   'direction_flip': {'caught': 14, 'n': 30},
   'object_swap': {'caught': 55, 'n':

## 4. The number that decides how you read nb9b

Headline recall is measured on a corrupted half whose error mix I chose. The mix that matters is the
one the *extractor* actually produces — and in this corpus that is overwhelmingly wrong properties
and types, not wrong entities. So the cell below scores each judge on the **22 real extractor errors**
in the clean half, and compares the grounding rate it would report against the human one.

That gap is the correction to apply to whatever nb9b prints.

In [10]:
gold, _, _, key, items = jb.gold_labels(GLOBAL_DIR)
by = {x["index"]: x for x in items}
real   = [i for i, k in key.items() if k["origin"] == "clean"]
truly_bad = [i for i in real if not gold[int(i)]]

print(f"On the {len(real)} REAL graph rows ({len(truly_bad)} of them genuinely unsupported):\n")
print(f"{'judge':38} {'judged grounded':>16} {'human':>8} {'caught real errors':>20}")
for t in jb.verdict_tags(GLOBAL_DIR):
    v = jb.load_json(GLOBAL_DIR / jb.VERDICTS_FILE.format(tag=t))["verdicts"]
    ok     = sum(1 for i in real if v[i]["supported"])
    caught = sum(1 for i in truly_bad if not v[i]["supported"])
    human  = sum(1 for i in real if gold[int(i)]) / len(real)
    print(f"{t:38} {ok/len(real):15.1%} {human:8.1%} {caught:>13}/{len(truly_bad)}"
          f" = {caught/len(truly_bad):.0%}")

print("\nreal extractor errors the FIRST run waved through:")
_v = jb.load_json(GLOBAL_DIR / jb.VERDICTS_FILE.format(tag=jb.verdict_tags(GLOBAL_DIR)[0]))["verdicts"]
for i in truly_bad:
    if _v[i]["supported"]:
        print(f'  [{i}] {by[int(i)]["assertion"].splitlines()[0][:92]}')
        print(f'        {by[int(i)]["human_note"][:92]}')

On the 250 REAL graph rows (22 of them genuinely unsupported):

judge                                   judged grounded    human   caught real errors
deepseek-v4-flash_p67305d45abb6_seed1234_run1           92.8%    91.2%            10/22 = 45%
deepseek-v4-flash_p67305d45abb6_seed1234_run2           90.8%    91.2%            10/22 = 45%
deepseek-v4-flash_p67305d45abb6_seed1234_run3           93.2%    91.2%            10/22 = 45%
llama33-70b_p67305d45abb6_seed1234_run1           92.0%    91.2%            12/22 = 55%
llama33-70b_p67305d45abb6_seed1234_run2           92.0%    91.2%            12/22 = 55%
llama33-70b_p67305d45abb6_seed1234_run3           92.0%    91.2%            12/22 = 55%
qwen3-32b-think_p67305d45abb6_seed1234_run1           92.8%    91.2%            12/22 = 55%
qwen3-32b-think_p67305d45abb6_seed1234_run2           93.2%    91.2%            12/22 = 55%
qwen3-32b-think_p67305d45abb6_seed1234_run3           92.0%    91.2%            12/22 = 55%
qwen3-32b-think_seed1234_run